# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access main metadata
meta = dataset.metadata
print(f"Title: {meta.name}\nDescription: {meta.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list all record sets and their fields referenced by their `@id`.

In [ ]:
# List all record sets (@id, name, description) and their fields (@id)
from pprint import pprint

record_sets = dataset.metadata.record_sets
if not record_sets:
    print('No record sets found in the Croissant schema.')
else:
    for rs in record_sets:
        print(f'Record set @id: {rs.id}')
        print(f'  Name: {rs.name}')
        print(f'  Description: {rs.description}')
        print('  Fields:')
        if hasattr(rs, 'fields') and rs.fields:
            for f in rs.fields:
                print(f'    - {f.id}')
        print('-' * 60)
    print(f"Total record sets: {len(record_sets)}")

For demonstration, let's list the first 3 records in the first available record set (if present):

In [ ]:
if record_sets:
    first_record_set = record_sets[0]
    print(f"Example records from record set '@id={first_record_set.id}':")
    for idx, rec in enumerate(dataset.records(record_set=first_record_set.id)):
        if idx >= 3:
            break
        pprint(rec)
else:
    print("No record sets to display records from.")

## 3. Data Extraction
Load data from all record sets into pandas DataFrames for processing. Entities are referenced by their `@id`.

In [ ]:
# Extract all record sets into DataFrames, dynamically using their @id
dfs = {}
rs_ids = [rs.id for rs in record_sets]

for rs_id in rs_ids:
    recs = list(dataset.records(record_set=rs_id))
    dfs[rs_id] = pd.DataFrame(recs) if recs else pd.DataFrame()

# Show columns from the first non-empty record set
for rs_id in rs_ids:
    if not dfs[rs_id].empty:
        print(f"Record set @id: {rs_id} columns: {dfs[rs_id].columns.tolist()}")
        display(dfs[rs_id].head())
        break

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter records, normalize numerical columns, group by categories, etc.

We'll select a numeric field and a categorical field by their `@id`. Adjust these values for your data!

In [ ]:
# Example: Assume first record set contains numeric and categorical data.

# Identify first usable record set and get field ids
main_rs_id = None
numeric_field = None
group_field = None
for rs in record_sets:
    df = dfs[rs.id]
    if not df.empty:
        main_rs_id = rs.id
        columns = df.columns.tolist()
        # Heuristically pick numeric and group fields by inspecting field names
        for c in columns:
            if 'abundance' in c.lower() or 'count' in c.lower() or 'mw' in c.lower() or 'coverage' in c.lower() or 'peptide' in c.lower():
                numeric_field = c
                break
        # Pick a group field that's not numeric
        for c in columns:
            if c != numeric_field and (c.lower().startswith('sample') or c.lower().startswith('type') or c.lower().startswith('group') or c.lower().startswith('accession') or c.lower().startswith('description')):
                group_field = c
                break
        break

if not main_rs_id or not numeric_field:
    print("No suitable numeric field for EDA found.")
else:
    df = dfs[main_rs_id]
    print(f"Using record set '@id': {main_rs_id}")
    print(f"Using numeric field (referenced by @id/col): {numeric_field}")
    if group_field:
        print(f"Using group-by field: {group_field}")

    # Ensure numeric
    df = df.copy()
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Set a threshold at the mean for demonstration
    threshold = df[numeric_field].mean()
    filtered = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered)} records.")

    # Normalize
    filtered[f"{numeric_field}_zscore"] = (filtered[numeric_field] - filtered[numeric_field].mean()) / filtered[numeric_field].std()
    print("First rows (numeric and normalized):")
    display(filtered[[numeric_field, f"{numeric_field}_zscore"]].head())

    # Group if possible
    if group_field and group_field in filtered.columns:
        grouped = filtered.groupby(group_field)[numeric_field].mean().reset_index()
        print("Grouped mean by group field:")
        display(grouped.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and, if grouped, show average values by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not main_rs_id or not numeric_field or dfs[main_rs_id].empty:
    print("No data to visualize.")
else:
    fig, ax = plt.subplots(1, 2 if group_field else 1, figsize=(12, 4))

    # Histogram of the numeric field
    sns.histplot(dfs[main_rs_id][numeric_field].dropna(), bins=20, ax=ax[0] if group_field else ax)
    ax[0].set_title(f"Distribution of {numeric_field}")

    # If grouped, a bar plot of group average
    if group_field:
        topn = grouped.sort_values(numeric_field, ascending=False).head(10)
        sns.barplot(data=topn, x=group_field, y=numeric_field, ax=ax[1])
        ax[1].set_title(f"Top groups mean {numeric_field}")
        ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we have loaded, examined, and visualized the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` Python library. Data was referenced and manipulated by each entity's `@id` for maximum interoperability. We performed filtering, normalization, and basic aggregation/group-wise analysis, along with initial visualizations of protein-related features captured by mass spectrometry. For deeper, domain-specific analysis, consult the dataset paper and Croissant documentation.